###Atividade 6 I.A Generativa em sinais de áudio

In [ ]:
# Instalação de dependências
!pip install -q git+https://github.com/openai/whisper.git
!pip install -q gtts gdown
!apt-get update -qq && apt-get install -y -qq ffmpeg
print('Instalação concluída')

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 3.4 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Instalação concluída


In [ ]:
# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive montado em /content/drive')

Mounted at /content/drive
Drive montado em /content/drive


In [ ]:
# 1) Verificar arquivos no ambiente e no Drive
print('Arquivos em /content:')
!ls -la /content | sed -n '1,200p'

print('\nArquivos em /content/drive/MyDrive (lista parcial):')
!ls -la /content/drive/MyDrive | sed -n '1,200p'

print('\nProcurando arquivos contendo "senna" no nome (pode demorar um pouco):')
!find /content /content/drive -maxdepth 4 -type f -iname "*senna*" -print | sed -n '1,200p'

Arquivos em /content:
total 20
drwxr-xr-x 1 root root 4096 Nov  4 00:54 .
drwxr-xr-x 1 root root 4096 Nov  4 00:50 ..
drwxr-xr-x 4 root root 4096 Oct 31 17:50 .config
drwx------ 4 root root 4096 Nov  4 00:54 drive
drwxr-xr-x 1 root root 4096 Oct 31 17:50 sample_data

Arquivos em /content/drive/MyDrive (lista parcial):
total 1744
drwx------ 2 root root   4096 May  7 22:35 Classroom
drwx------ 2 root root   4096 Jun  3 19:56 Colab Notebooks
-rw------- 1 root root 554430 Oct 11 16:43 CV - Wesley R. Silva (1).pdf
-rw------- 1 root root 367024 Sep 21 00:34 Trabalho_teste05.ipynb
-rw------- 1 root root 245047 Oct  8 01:16 Trabalho_versao_Final.ipynb
-rw------- 1 root root 340916 Sep 15 23:21 Untitled0.ipynb
-rw------- 1 root root 249227 Sep 21 23:19 Versão didática projeto (não enviar ao Professor essa versão).ipynb
-rw------- 1 root root  20100 Sep 16 00:05 Wesley_Rafael_da_Silva–RA625105397.csv

Procurando arquivos contendo "senna" no nome (pode demorar um pouco):


In [ ]:
# 3) Fazer upload manual do arquivo a partir do seu computador
# Execute esta célula se preferir enviar o arquivo diretamente para o ambiente do Colab.
from google.colab import files
uploaded = files.upload()
if uploaded:
    for name in uploaded.keys():
        print('Upload concluído:', name, '-> /content/' + name)

Saving audio_entrevista_Senna.wav to audio_entrevista_Senna.wav
Upload concluído: audio_entrevista_Senna.wav -> /content/audio_entrevista_Senna.wav


In [ ]:
# detectar upload -> converter -> transcrever -> extrair Q&A -> gerar TTS
import os, glob, shlex, subprocess, time, traceback
from pathlib import Path

# ---------- 1) localizar uploads recentes em /content ----------
candidates = list(glob.glob('/content/*'))
# remover pastas que não sejam arquivos de mídia comuns (mantemos apenas arquivos)
candidates = [p for p in candidates if os.path.isfile(p)]
# filtrar extensões de áudio comuns
audio_exts = {'.mp3', '.m4a', '.wav', '.ogg', '.webm', '.flac', '.aac'}
audio_files = [p for p in candidates if Path(p).suffix.lower() in audio_exts]

if not audio_files:
    raise FileNotFoundError("Nenhum arquivo de áudio encontrado em /content. Faça upload via files.upload() e tente novamente.")

# escolher o arquivo mais recentemente modificado
audio_path = max(audio_files, key=lambda p: os.path.getmtime(p))
print("Arquivo detectado para processamento:", audio_path, "- tamanho:", round(os.path.getsize(audio_path)/(1024*1024),2), "MB")

# ---------- 2) converter para WAV 16k mono (opcional, mas ajuda) ----------
converted = '/content/entrevista_senna_converted.wav'
def convert_to_wav(src, dst):
    cmd = ['ffmpeg', '-y', '-i', src, '-ac', '1', '-ar', '16000', '-vn', dst]
    print("Convertendo para WAV 16k mono (pode demorar):", ' '.join(shlex.quote(x) for x in cmd))
    try:
        subprocess.run(cmd, check=True, capture_output=True)
        print("Conversão concluída:", dst)
        return dst
    except subprocess.CalledProcessError as e:
        print("Conversão falhou. Tentarei usar o arquivo original diretamente. ffmpeg stdout/stderr abaixo:")
        try:
            print("STDOUT:", e.stdout.decode(errors='ignore'))
            print("STDERR:", e.stderr.decode(errors='ignore'))
        except Exception:
            pass
        return None

# se já for WAV 16k mono, podemos pular; mas vamos tentar converter independentemente
conv_result = convert_to_wav(audio_path, converted)
if conv_result and os.path.exists(conv_result):
    chosen_audio = conv_result
else:
    print("Usando arquivo original (Whisper também aceita mp3/m4a):", audio_path)
    chosen_audio = audio_path

# ---------- 3) Transcrever com Whisper ----------
print("\nIniciando transcrição com Whisper (modelo small). Isso pode levar alguns minutos.")
try:
    import whisper
    model = whisper.load_model("small")
    result = model.transcribe(chosen_audio, word_timestamps=False)
except Exception:
    traceback.print_exc()
    raise RuntimeError("Falha ao transcrever com Whisper. Veja a saída acima para detalhes (possível problema de ffmpeg/codec).")

# salvar transcrição (prefere Drive se montado)
drive_path_txt = '/content/drive/MyDrive/entrevista_senna_transcricao.txt'
local_path_txt = '/content/entrevista_senna_transcricao.txt'
out_path_txt = drive_path_txt if os.path.exists('/content/drive/MyDrive') else local_path_txt
with open(out_path_txt, 'w', encoding='utf-8') as f:
    f.write(result.get('text',''))
print("Transcrição salva em:", out_path_txt)

# extrair segmentos
segments = result.get('segments', [])
print(f"{len(segments)} segmentos detectados. Exibindo primeiras 30 (se houver):\n")
for i, s in enumerate(segments[:30]):
    print(f"[{i}] {s['start']:.2f}s - {s['end']:.2f}s : {s['text'].strip()}\n")

segments_list = segments  # variável usada pela etapa QA

# ---------- 4) Extração automática de Q&A (heurística PT) ----------
import re, pandas as pd, json
from pathlib import Path
from IPython.display import display, Audio

MIN_QUESTION_LENGTH = 3
MAX_ANSWER_SECONDS = 60 * 5
OUT_CSV = '/content/drive/MyDrive/entrevista_senna_qa_pairs.csv' if os.path.exists('/content/drive/MyDrive') else '/content/entrevista_senna_qa_pairs.csv'
SYNTHESIZE_ANSWERS = True
TTS_OUT_FOLDER = '/content/drive/MyDrive/tts_respostas_auto_answers' if os.path.exists('/content/drive/MyDrive') else '/content/tts_respostas_auto_answers'
TTS_LANG = 'pt'

def is_question_pt(text):
    if not text or len(text.strip()) < MIN_QUESTION_LENGTH:
        return False
    txt = text.strip()
    if '?' in txt:
        return True
    q_words = [
        r'^\s*quem\b', r'^\s*o que\b', r'^\s*oque\b', r'^\s*qual\b', r'^\s*quais\b',
        r'^\s*quando\b', r'^\s*onde\b', r'^\s*por que\b', r'^\s*porquê\b',
        r'^\s*como\b', r'^\s*quanto\b'
    ]
    for pat in q_words:
        if re.search(pat, txt, flags=re.IGNORECASE):
            return True
    impl = [r'\bpoderia\b', r'\bgostaria de saber\b', r'\bqueria saber\b']
    for pat in impl:
        if re.search(pat, txt, flags=re.IGNORECASE):
            return True
    return False

if not segments_list:
    print("AVISO: nenhum segmento detectado pela transcrição. A extração Q&A pode não gerar resultados.")
qa_pairs = []
i = 0; n = len(segments_list)
while i < n:
    seg = segments_list[i]
    text = seg.get('text','').strip()
    if is_question_pt(text):
        question = text
        answer_texts = []
        j = i + 1
        start_time = segments_list[j].get('start',0) if j < n else seg.get('end',0)
        while j < n:
            cand = segments_list[j]
            cand_text = cand.get('text','').strip()
            cand_end = cand.get('end', cand.get('start',0))
            if is_question_pt(cand_text):
                break
            if (cand_end - start_time) > MAX_ANSWER_SECONDS:
                break
            answer_texts.append(cand_text)
            j += 1
        answer = ' '.join(t for t in answer_texts if t)
        if not answer and i+1 < n:
            answer = segments_list[i+1].get('text','').strip()
            j = i+2
        qa_pairs.append({
            'question_index': i,
            'question_start': seg.get('start',0),
            'question_end': seg.get('end',0),
            'question': question,
            'answer_start': segments_list[i+1].get('start', None) if i+1<n else None,
            'answer_end': segments_list[j-1].get('end', None) if (j-1)<n else None,
            'answer': answer,
            'answer_segment_indices': list(range(i+1,j))
        })
        i = j
    else:
        i += 1

df = pd.DataFrame(qa_pairs)
for col in ['question','answer']:
    if col in df.columns:
        df[col] = df[col].str.replace(r'\s+', ' ', regex=True).str.strip()

Path(OUT_CSV).parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_CSV, index=False, encoding='utf-8-sig')
print(f"Extração concluída: {len(df)} pares Q&A detectados. CSV salvo em: {OUT_CSV}")
display(df.head(30))

# salvar json
json_out = str(Path(OUT_CSV).with_suffix('.json'))
with open(json_out,'w',encoding='utf-8') as jf:
    json.dump(qa_pairs, jf, ensure_ascii=False, indent=2)
print("JSON salvo em:", json_out)

# ---------- 5) Gerar TTS das respostas encontradas (gTTS) ----------
if SYNTHESIZE_ANSWERS and not df.empty:
    try:
        from gtts import gTTS
        os.makedirs(TTS_OUT_FOLDER, exist_ok=True)
        tts_paths = []
        for idx, row in df.iterrows():
            ans_text = row.get('answer','').strip()
            if not ans_text:
                continue
            fname = f'answer_{str(idx).zfill(2)}.mp3'
            fpath = str(Path(TTS_OUT_FOLDER) / fname)
            gTTS(text=ans_text, lang=TTS_LANG, slow=False).save(fpath)
            tts_paths.append(fpath)
            print("Gerado TTS:", fpath)
        if tts_paths:
            print("\nReproduzindo os primeiros 5 audios de resposta (inline):")
            for pth in tts_paths[:5]:
                display(Audio(pth, autoplay=False))
    except Exception as e:
        print("Falha ao sintetizar respostas (gTTS). Erro:", e)

print("\nPROCESSO FINALIZADO. Verifique os arquivos gerados no seu Drive (ou em /content).")


Arquivo detectado para processamento: /content/audio_entrevista_Senna.wav - tamanho: 13.65 MB
Convertendo para WAV 16k mono (pode demorar): ffmpeg -y -i /content/audio_entrevista_Senna.wav -ac 1 -ar 16000 -vn /content/entrevista_senna_converted.wav
Conversão concluída: /content/entrevista_senna_converted.wav

Iniciando transcrição com Whisper (modelo small). Isso pode levar alguns minutos.


100%|████████████████████████████████████████| 461M/461M [00:04<00:00, 112MiB/s]
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Transcrição salva em: /content/drive/MyDrive/entrevista_senna_transcricao.txt
12 segmentos detectados. Exibindo primeiras 30 (se houver):

[0] 0.00s - 8.00s : Nosso carro no momento tem alguns problemas que ficam maiores em pistas mais lentas e onduladas.

[1] 8.00s - 16.00s : Naturalmente com o motor Renault que tem uma boa potência, pistas mais velozes nos dão alguma vantagem sobre o Benetton,

[2] 16.00s - 18.00s : que é o carro mais competitivo do momento.

[3] 18.00s - 26.00s : Mas por outro lado, os problemas que nós já identificamos no Brasil e aqui estão sendo trabalhados no momento na Inglaterra pela equipe

[4] 26.00s - 33.00s : e eu espero que já em Imola a gente tenha algumas modificações no Chassis que permitam então uma melhoria na performance do Chassis.

[5] 33.00s - 36.00s : E queremos que a matipulação da suspensão da frente com o aerodinimismo?

[6] 36.00s - 47.00s : Não, é uma combinação, é o conjunto de aerodinâmica e suspensão, mas é um problema que ocorre sempre.

,question_index,question_start,question_end,question,answer_start,answer_end,answer,answer_segment_indices
0,5,33.0,36.0,E queremos que a matipulação da suspensão da f...,36.0,47.0,"Não, é uma combinação, é o conjunto de aerodin...",[6]
1,7,47.0,53.0,Quando se muda um regulamento drasticamente e ...,53.0,80.0,a gente tem um risco sem muito grande de algum...,"[8, 9, 10, 11]"


JSON salvo em: /content/drive/MyDrive/entrevista_senna_qa_pairs.json
Gerado TTS: /content/drive/MyDrive/tts_respostas_auto_answers/answer_00.mp3
Gerado TTS: /content/drive/MyDrive/tts_respostas_auto_answers/answer_01.mp3

Reproduzindo os primeiros 5 audios de resposta (inline):



PROCESSO FINALIZADO. Verifique os arquivos gerados no seu Drive (ou em /content).
